### Harmonization of the metadata of the remapped studies
- **Developed by:** Anna Maguza
- **Affilation:** Faculty of Medicine, Würzburg University
- **Creation date:** 8th of November 2024
- **Last modified date:** 16th of December 2024

This notebooks manually harmonize metadata columns and values. Here we process such studies:
* E-MTAB-8901 (Elementaite et al, 2021) - fetal developing gut
* E-MTAB-9536 (Elementaite et al, 2021) - fetal developing gut
* E-MTAB-9543 (Elementaite et al, 2021) - adult gut
* E-MTAB-9489 (Holloway et al, 2021) - fetal developing gut
* E-MTAB-9720 (Holloway et al, 2021) - fetal enteroids

### Import packages

In [31]:
import pandas as pd
from typing import Dict, List, Optional, Union
import os
import json
import scanpy as sc
import re
from datetime import datetime
import numpy as np

### Define functions

In [32]:
def update_time_unit(row):
    if row['developmental_stage'] in ['embryonic human stage', '9th week post-fertilization human stage', '10th week post-fertilization human stage']:
        return 'week'
    else:
        return 'year'

In [33]:
def transform_column_name(col):
    if any(col.startswith(prefix) for prefix in ['Comment[', 'Characteristics[', 'Unit[', 'Factor Value[']) and col.endswith(']'):
        new_name = col[col.find('[')+1:-1]
        new_name = new_name.replace(' ', '_')
        return new_name
    return col

In [34]:
def add_space_between_number_and_letter(age_str):
    return re.sub(r'(\d)([a-zA-Z])', r'\1 \2', age_str)

In [35]:
def determine_trimester(age_str):
    try:
        weeks = float(age_str.split()[0])
        if weeks <= 12:
            return 'first trimester'
        elif weeks <= 26:
            return 'second trimester'
        else:
            return 'third trimester'
    except:
        return 'NA'

### Load data

In [36]:
E8901 = sc.read_h5ad('Elementaite_2021/gut_hs_Elementaite2021_E-MTAB-8901_QC_filtered_AM_02122024_143702_raw.h5ad')
E8901

AnnData object with n_obs × n_vars = 131623 × 70711
    obs: 'sample_name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[developmental stage]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[gestational age]', 'Unit[time unit].1', 'Term Source REF.1', 'Term Accession Number.1', 'Characteristics[sex]', 'Characteristics[disease]', 'Characteristics[individual]', 'Characteristics[organism part]', 'Characteristics[cell type]', 'Characteristics[immunophenotype]', 'Characteristics[growth condition]', 'Characteristics[passage]', 'Material Type', 'Protocol REF', 'Protocol REF.1', 'Protocol REF.2', 'Protocol REF.3', 'Protocol REF.4', 'Extract Name', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATIO

In [37]:
E9536 = sc.read_h5ad('Elementaite_2021/gut_hs_Elementaite2021_E-MTAB-9536_QC_filtered_AM_13112024_104420_raw.h5ad')
E9536

AnnData object with n_obs × n_vars = 31494 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[individual]', 'Characteristics[disease]', 'Characteristics[organism part]', 'Characteristics[sampling site]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[lib

In [38]:
E9543 = sc.read_h5ad('Elementaite_2021/gut_hs_Elementaite2021_E-MTAB-9543_QC_filtered_AM_09122024_154541_raw.h5ad')
E9543

AnnData object with n_obs × n_vars = 62586 × 70711
    obs: 'Extract Name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[developmental stage]', 'Characteristics[sex]', 'Characteristics[individual]', 'Characteristics[cell type]', 'Characteristics[organism part]', 'Material Type', 'Protocol REF', 'Performer', 'Protocol REF.1', 'Performer.1', 'Protocol REF.2', 'Performer.2', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRAND]', 'Comment[LIBRARY_STRATEGY]', 'Comment[NOMINAL_LENGTH]', 'Comment[NOMINAL_SDEV]', 'Comment[ORIENTATION]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library con

In [39]:
E9489 = sc.read_h5ad('Holloway_2021/gut_hs_Holloway2021_E-MTAB-9489_QC_filtered_AM_09122024_100120_raw.h5ad')
E9489

AnnData object with n_obs × n_vars = 223842 × 70711
    obs: 'sample_name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[developmental stage]', 'Characteristics[age]', 'Unit[time unit]', 'Term Source REF', 'Term Accession Number', 'Characteristics[sex]', 'Characteristics[disease]', 'Characteristics[individual]', 'Characteristics[organism part]', 'Characteristics[immunophenotype]', 'Material Type', 'Protocol REF', 'Protocol REF.1', 'Protocol REF.2', 'Protocol REF.3', 'Extract Name', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRATEGY]', 'Comment[LIBRARY_STRAND]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library construction]', 'Comment[primer]', 'Comment[sample bar

In [40]:
E9720 = sc.read_h5ad('Holloway_2021/gut_hs_Holloway2021_E-MTAB-9720_QC_filtered_AM_09122024_103213_raw.h5ad')
E9720

AnnData object with n_obs × n_vars = 10558 × 70711
    obs: 'sample_name', 'batch', 'barcode', 'Source Name', 'Comment[ENA_SAMPLE]', 'Comment[BioSD_SAMPLE]', 'Characteristics[organism]', 'Characteristics[disease]', 'Characteristics[genotype]', 'Characteristics[organism part]', 'Characteristics[cell type]', 'Characteristics[growth condition]', 'Characteristics[developmental stage]', 'Material Type', 'Protocol REF', 'Protocol REF.1', 'Protocol REF.2', 'Protocol REF.3', 'Extract Name', 'Comment[LIBRARY_LAYOUT]', 'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]', 'Comment[LIBRARY_STRATEGY]', 'Comment[cdna read]', 'Comment[cdna read offset]', 'Comment[cdna read size]', 'Comment[cell barcode offset]', 'Comment[cell barcode read]', 'Comment[cell barcode size]', 'Comment[end bias]', 'Comment[input molecule]', 'Comment[library construction]', 'Comment[primer]', 'Comment[sample barcode offset]', 'Comment[sample barcode read]', 'Comment[sample barcode size]', 'Comment[single cell isolation]

### Add study name

In [41]:
E9536.obs['Study_name'] = 'Elementaite_2021'
E8901.obs['Study_name'] = 'Elementaite_2021'
E9543.obs['Study_name'] = 'Elementaite_2021'
E9489.obs['Study_name'] = 'Holloway_2021'
E9720.obs['Study_name'] = 'Holloway_2021'

In [42]:
E9536.obs['ArrayExpress_ID'] = 'E-MTAB-9536'
E8901.obs['ArrayExpress_ID'] = 'E-MTAB-8901'
E9543.obs['ArrayExpress_ID'] = 'E-MTAB-9543'
E9489.obs['ArrayExpress_ID'] = 'E-MTAB-9489'
E9720.obs['ArrayExpress_ID'] = 'E-MTAB-9720'

### Harmonize columns present in all objects

In [44]:
anndata_objects = [E8901, E9489, E9536, E9543, E9720]

common_columns = set(anndata_objects[0].obs.columns)

for adata in anndata_objects[1:]:
    common_columns.intersection_update(adata.obs.columns)

print("Common columns in obs across all AnnData objects:", common_columns)

Common columns in obs across all AnnData objects: {'Comment[LIBRARY_SOURCE]', 'Protocol REF.3', 'Comment[library construction]', 'Comment[FASTQ_URI]', 'Comment[umi barcode offset]', 'predicted_doublets', 'Characteristics[organism]', 'Source Name', 'Material Type', 'percent_ribo', 'Comment[primer]', 'log1p_n_genes', 'percent_top50', 'Comment[cdna read size]', 'Comment[input molecule]', 'Protocol REF.2', 'batch', 'percent_hb', 'Study_name', 'Comment[umi barcode size]', 'XIST-percentage', 'Comment[LIBRARY_LAYOUT]', 'Technology Type', 'Cell_cycle_phase', 'n_counts', 'Protocol REF.1', 'Extract Name', 'Comment[spike in]', 'Comment[BioSD_SAMPLE]', 'sex', 'doublet_scores', 'Comment[cdna read]', 'Comment[cell barcode read]', 'Performer', 'Scan Name', 'log1p_n_counts', 'n_counts_mito', 'total_counts', 'ArrayExpress_ID', 'Comment[SUBMITTED_FILE_NAME]', 'Protocol REF', 'cell_passed_qc', 'percent_mito', 'Comment[sample barcode size]', 'barcode', 'Comment[LIBRARY_SELECTION]', 'Comment[umi barcode re

In [45]:
numeric_columns = []
string_columns = []

for col in common_columns:
    if pd.api.types.is_numeric_dtype(E8901.obs[col]):
        numeric_columns.append(col)
    else:
        string_columns.append(col)

print("Numeric columns:", numeric_columns)
print("String columns:", string_columns)

Numeric columns: ['Comment[umi barcode offset]', 'percent_ribo', 'log1p_n_genes', 'percent_top50', 'Comment[cdna read size]', 'percent_hb', 'Comment[umi barcode size]', 'XIST-percentage', 'n_counts', 'doublet_scores', 'log1p_n_counts', 'n_counts_mito', 'total_counts', 'cell_passed_qc', 'percent_mito', 'Comment[sample barcode size]', 'Comment[sample barcode offset]', 'percent_chrY', 'XIST-counts', 'Comment[cell barcode size]', 'n_genes_by_counts', 'n_counts_hb', 'S_score', 'Comment[cdna read offset]', 'G2M_score', 'n_genes', 'n_counts_ribo', 'Comment[cell barcode offset]']
String columns: ['Comment[LIBRARY_SOURCE]', 'Protocol REF.3', 'Comment[library construction]', 'Comment[FASTQ_URI]', 'predicted_doublets', 'Characteristics[organism]', 'Source Name', 'Material Type', 'Comment[primer]', 'Comment[input molecule]', 'Protocol REF.2', 'batch', 'Study_name', 'Comment[LIBRARY_LAYOUT]', 'Technology Type', 'Cell_cycle_phase', 'Protocol REF.1', 'Extract Name', 'Comment[spike in]', 'Comment[BioS

#### Harmonize numeric columns that are present in all datasets

In [46]:
avg_dict = {col: [] for col in numeric_columns}

for adata in anndata_objects:
    for col in numeric_columns:
        if col in adata.obs.columns:
            try:
                values = pd.to_numeric(adata.obs[col], errors='coerce')
                col_avg = values.mean()
                avg_dict[col].append(col_avg)
            except (TypeError, ValueError):
                avg_dict[col].append(None)
        else:
            avg_dict[col].append(None)

avg_df = pd.DataFrame(avg_dict, index=[f"{adata.obs['ArrayExpress_ID'][0]}" for adata in anndata_objects])

avg_df = avg_df.T

/tmp/ipykernel_493254/60154284.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  avg_df = pd.DataFrame(avg_dict, index=[f"{adata.obs['ArrayExpress_ID'][0]}" for adata in anndata_objects])


In [47]:
columns_to_delete = [
    'Comment[sample barcode offset]', 
    'Comment[cdna read offset]', 
    'Comment[cell barcode offset]'
]

for adata in anndata_objects:
    adata.obs.drop(columns=columns_to_delete, inplace=True)

In [48]:
avg_df

,E-MTAB-8901,E-MTAB-9489,E-MTAB-9536,E-MTAB-9543,E-MTAB-9720
Comment[umi barcode offset],16.000000,16.000000,16.000000,16.000000,16.000000
percent_ribo,40.642932,28.183741,31.651336,3.126205,23.655421
log1p_n_genes,6.963064,7.037464,7.102400,5.721725,7.782805
percent_top50,42.015546,34.872323,37.244403,64.894935,40.247578
Comment[cdna read size],98.000000,96.424326,98.000000,98.000000,91.000000
percent_hb,0.016510,0.018333,0.015790,0.023896,0.000034
Comment[umi barcode size],10.000000,10.450193,10.000000,10.000000,12.000000
XIST-percentage,0.002627,0.004614,0.002350,0.002629,0.001902
n_counts,5361.579656,4468.988108,14192.733568,985.882610,11698.933510
doublet_scores,0.163471,0.112941,0.148045,0.137476,0.179248


In [49]:
for adata in anndata_objects:
    rename_dict = {}
    
    for col in numeric_columns:
        if col in adata.obs.columns and col.startswith('Comment['):
            new_name = transform_column_name(col)
            rename_dict[col] = new_name
    
    if rename_dict:
        adata.obs.rename(columns=rename_dict, inplace=True)

#### Harmonize string columns that are present in all datasets

In [50]:
analysis_data = []
for col in string_columns:
    for adata in anndata_objects:
        dataset_id = adata.obs['ArrayExpress_ID'].iloc[0]
        
        if col in adata.obs.columns:
            unique_values = adata.obs[col].unique()
            example_values = adata.obs[col].iloc[:5]
            
            analysis_data.append({
                'Column': col,
                'Dataset_ID': dataset_id,
                'N_unique_values': len(unique_values),
                'Unique_values': ', '.join(map(str, unique_values)),
                'Example_values': ', '.join(map(str, example_values))
            })
        else:
            analysis_data.append({
                'Column': col,
                'Dataset_ID': dataset_id,
                'N_unique_values': 0,
                'Unique_values': 'Column not present',
                'Example_values': 'N/A'
            })

analysis_df = pd.DataFrame(analysis_data)

In [51]:
columns_to_delete = [
    #'Comment[FASTQ_URI]', 
    'Protocol REF.1', 
    'Technology Type', # all values are 'sequencing assay'
    'Comment[LIBRARY_STRATEGY]', # all values are 'RNA-Seq'
    'Comment[cell barcode read]', # all values are 'read 1'
    'Comment[LIBRARY_SELECTION]', # all values are 'Oligo-dT'
    'Comment[umi barcode read]', # all values are 'read 1'
    'Comment[primer]', # all values are 'oligo-dT'
    'Protocol REF.2', 
    'Comment[sample barcode read]', # all values are 'index 1'
    'Scan Name',
    'Comment[spike in]', # all values are 'none'
    'Comment[single cell isolation]', # all values are '10x'
    'Protocol REF.3',
    'Comment[SUBMITTED_FILE_NAME]',
    'Comment[cdna read]', # all values are 'read 2'
    'Comment[LIBRARY_SOURCE]', # all values are 'TRANSCRIPTOMIC SINGLE CELL'
    'Comment[input molecule]',  # all values are 'polyA RNA'
    'batch' # column arised when i merged the datasets, dont represent real batch

]

In [52]:
for adata in anndata_objects:
    adata.obs.drop(columns=columns_to_delete, inplace=True)

In [53]:
for adata in anndata_objects:
    rename_dict = {}
    for col in string_columns:
        if (col in adata.obs.columns and 
            any(col.startswith(prefix) for prefix in ['Comment[', 'Characteristics[', 'Unit['])):
            new_name = transform_column_name(col)
            rename_dict[col] = new_name
    if rename_dict:
        adata.obs.rename(columns=rename_dict, inplace=True)

#### Add sample_id

In [54]:
for adata in anndata_objects:
    adata.obs.rename(columns={'Extract Name': 'sample_id'}, inplace=True)

#### Add library_preparation_protocol

In [55]:
for adata in anndata_objects:
    adata.obs['library_construction'] = adata.obs['library_construction'].astype(str)
    adata.obs['end_bias'] = adata.obs['end_bias'].astype(str)
    adata.obs['library_preparation_protocol'] = adata.obs['library_construction'] + ' ' + adata.obs['end_bias']

#### Add batch

In [65]:
E8901.obs['batch'] = E8901.obs['FASTQ_URI'].str.split('/').str[-1].str.replace('_S1_L001_R1_001.fastq.gz', '', regex=False)
E8901.obs['batch'].value_counts()

batch
4918STDY7901096    11818
4918STDY7693757     7198
4918STDY7333456     5152
4918STDY7317585     5003
4918STDY7717783     4411
4918STDY8366754     4394
4918STDY7273964     4263
4918STDY7426910     4168
4918STDY7717785     3943
4918STDY7982703     3621
4918STDY7421298     3445
4918STDY7447825     3392
4918STDY7717784     3351
4918STDY7693758     3285
4918STDY7426906     3284
4918STDY7421297     3110
4918STDY7321515     3027
4918STDY7273965     2951
4918STDY7426911     2745
4918STDY7962470     2704
4918STDY7693759     2699
4918STDY7317587     2689
4918STDY7426908     2590
4918STDY7923744     2403
4918STDY7693760     2227
4918STDY7426905     1993
4918STDY7844898     1977
4918STDY7321514     1938
4918STDY7693763     1846
4918STDY7590325     1821
4918STDY7426907     1753
4918STDY7426904     1722
4918STDY7717789     1721
4918STDY7982702     1682
4918STDY7317586     1680
4918STDY7693762     1596
4918STDY7321513     1463
4918STDY7844899     1459
4918STDY8366756     1454
4918STDY7693761    

In [61]:
E9536.obs['batch'] = E9536.obs['FASTQ_URI'].str.split('/').str[-1].str.replace('_S1_L001_R1_001.fastq.gz', '', regex=False)
E9536.obs['batch'].value_counts()

batch
Human_colon_16S7985389    6400
Human_colon_16S7985391    4688
Human_colon_16S7985393    4461
Human_colon_16S7985392    4372
Human_colon_16S7985394    4291
Human_colon_16S7985390    3603
Human_colon_16S7985395    2122
Human_colon_16S8159183     299
Human_colon_16S8159185     175
Human_colon_16S8159186     155
Human_colon_16S8159189     153
Human_colon_16S8159187     152
Human_colon_16S8159190     150
Human_colon_16S8159188     113
FCA_gut8015057              61
Human_colon_16S8159182      58
FCA_gut8015058              55
Human_colon_16S8159184      54
FCA_gut8015061              51
FCA_gut8015060              45
FCA_gut8015059              36
Name: count, dtype: int64

In [64]:
E9543.obs['batch'] = E9543.obs['FASTQ_URI'].str.split('/').str[-1].str.replace('_S1_L001_I1_001.fastq.gz', '', regex=False)
E9543.obs['batch'].value_counts()

batch
Human_colon_16S8001903    5604
Human_colon_16S8117831    5340
Human_colon_16S8117829    4836
Human_colon_16S8000513    4725
Human_colon_16S8117830    3903
Human_colon_16S8117828    3537
Human_colon_16S8002624    2286
Human_colon_16S8002629    1635
Human_colon_16S8001907    1602
Human_colon_16S8002630    1581
Human_colon_16S8159194    1353
Human_colon_16S8002627    1335
Human_colon_16S8001905    1284
Human_colon_16S8159191    1188
Human_colon_16S8159193    1125
Human_colon_16S8159192    1065
Human_colon_16S8002626    1044
Human_colon_16S8002628    1041
Human_colon_16S8123913     867
Human_colon_16S8001865     831
Human_colon_16S8001871     789
Human_colon_16S8123910     786
Human_colon_16S8000515     786
Human_colon_16S8000473     777
Human_colon_16S8002623     744
Human_colon_16S8123919     723
Human_colon_16S8123911     720
Human_colon_16S8123916     717
Human_colon_16S8000481     711
Human_colon_16S8001878     687
Human_colon_16S8000511     684
Human_colon_16S8001885     639
Hu

### Normalize columns that are not present in every dataset

In [67]:
common_columns = set(anndata_objects[0].obs.columns)

for adata in anndata_objects[1:]:
    common_columns.intersection_update(adata.obs.columns)

In [68]:
all_columns = set()
for adata in anndata_objects:
    all_columns.update(set(adata.obs.columns) - set(common_columns))

analysis_data = []

for col in all_columns:
    for adata in anndata_objects:
        dataset_id = adata.obs['ArrayExpress_ID'].iloc[0]
        
        if col in adata.obs.columns:
            unique_values = adata.obs[col].unique()
            example_values = adata.obs[col].iloc[:5]
            
            analysis_data.append({
                'Column': col,
                'Dataset_ID': dataset_id,
                'Present': 'Yes',
                'N_unique_values': len(unique_values),
                'Unique_values': ', '.join(map(str, unique_values)),
                'Example_values': ', '.join(map(str, example_values))
            })
        else:
            analysis_data.append({
                'Column': col,
                'Dataset_ID': dataset_id,
                'Present': 'No',
                'N_unique_values': 0,
                'Unique_values': 'Not present',
                'Example_values': 'N/A'
            })

analysis_df = pd.DataFrame(analysis_data)

In [69]:
columns_to_delete = [
    'Protocol REF.4', 
    'FASTQ_URI',
    'Comment[NOMINAL_LENGTH]', 
    'Comment[read1 file]', 
    'Term Source REF.1', 
    'Protocol REF.5', 
    'Comment[index1 file]', 
    'Performer.3', 
    'Comment[technical replicate group]', # same as sample_id
    'Characteristics[genotype]', 
    'Comment[ORIENTATION]', # same as end_bias
    'Comment[read_type]',
    'Term Accession Number.1', 
    'Comment[BAM_URI]', 
    'Comment[FASTQ_URI2]',
    'sample_name', 
    'Comment[read2 file]', 
    'Factor Value[developmental stage]',
    'Unit[time unit].1',  
    'Performer.2',
    'Comment[read_index]',
    'Comment[FASTQ_URI].1',
    'Factor Value[single cell identifier]',
    'Comment[LIBRARY_STRAND]',
    'Term Accession Number',
    'Comment[FASTQ_URI].2',
    'Comment[NOMINAL_SDEV]',
    'Term Source REF',
    'Performer.1',
    'Comment[FASTQ_URI3]',
    'Factor Value[disease]',
    'Factor Value[sampling site]',
    'Factor Value[organism part]' # same as Characteristics[organism part]
]

In [70]:
for adata in anndata_objects:
    columns_to_drop = [col for col in columns_to_delete if col in adata.obs.columns]
    if columns_to_drop:
        adata.obs.drop(columns=columns_to_drop, inplace=True)

### Fix age

+ E8901

In [71]:
E8901.obs['time_unit'] = E8901.obs.apply(update_time_unit, axis=1)

In [72]:
E8901.obs['Characteristics[age]'] = E8901.obs['Characteristics[age]'].astype(str)
E8901.obs['Characteristics[gestational age]'] = E8901.obs['Characteristics[gestational age]'].astype(str)
E8901.obs['time_unit'] = E8901.obs['time_unit'].astype(str)

In [73]:
E8901.obs['full_age'] = E8901.obs['Characteristics[age]'] + ' ' + E8901.obs['Characteristics[gestational age]'] + ' ' + E8901.obs['time_unit']
E8901.obs['full_age'] = E8901.obs['full_age'].str.replace(' ', '')


E8901.obs['full_age'] = E8901.obs['full_age'].apply(add_space_between_number_and_letter)

E8901.obs['full_age'].value_counts()

full_age
11.9 week    15474
11.2 week    15028
11 year      11818
10.4 week     9372
12.2 week     7559
12 year       7444
4 year        7125
8.7 week      6999
8.4 week      6913
8.1 week      6428
8.5 week      5848
8.9 week      5296
9.9 week      5149
9 year        5107
13 year       4910
6 year        3392
7.4 week      3126
10 year       2953
14 year       1682
Name: count, dtype: int64

+ E9536

In [74]:
E9536.obs['Characteristics[age]'] = E9536.obs['Characteristics[age]'].astype(str)
E9536.obs['time_unit'] = E9536.obs['time_unit'].astype(str)
E9536.obs['full_age'] = E9536.obs['Characteristics[age]'] + ' ' + E9536.obs['time_unit']
E9536.obs['full_age'].value_counts()

full_age
15 week    19631
12 week    10874
16 week      741
17 week      248
Name: count, dtype: int64

+ E9543

In [75]:
E9543.obs['Characteristics[age]'] = E9543.obs['Characteristics[age]'].astype(str)
E9543.obs['time_unit'] = E9543.obs['time_unit'].astype(str)
E9543.obs['full_age'] = E9543.obs['Characteristics[age]'] + ' ' + E9543.obs['time_unit']
E9543.obs['full_age'].value_counts()

full_age
25 to 30 year    20706
70 to 75 year    17616
60 to 65 year    10212
65 to 70 year     8781
45 to 50 year     5271
Name: count, dtype: int64

+ E9489

In [76]:
E9489.obs['full_age'] = (E9489.obs['Characteristics[age]'] / 7).round(1).astype(str) + ' week'
E9489.obs['full_age'].value_counts()

full_age
14.4 week    51040
17.4 week    48160
10.3 week    27386
8.4 week     23000
18.9 week    22867
18.1 week    21318
6.7 week     16934
11.4 week    13137
Name: count, dtype: int64

+ E9720

In [77]:
E9720.obs['full_age'] = (E9720.obs['Factor Value[time]'] / 7).round(1).astype(str) + ' week'
E9720.obs['full_age'].value_counts()

full_age
15.0 week    6609
20.3 week    3949
Name: count, dtype: int64

### Add disease information where absent

In [78]:
E9543.obs['Characteristics[disease]'] = 'normal'

### Add information about organoids

In [79]:
E9489.obs['Characteristics[growth condition]'] = 'primary tissue'
E9543.obs['Characteristics[growth condition]'] = 'primary tissue'
E9536.obs['Characteristics[growth condition]'] = 'primary tissue'

In [80]:
E9720.obs['Factor Value[stimulus]'] = E9720.obs['Factor Value[stimulus]'].astype(str)
E9720.obs['Characteristics[growth condition]'] = 'fetal enteroid grown medium' + ' ' + E9720.obs['Factor Value[stimulus]']
E9720.obs['Characteristics[growth condition]'].value_counts()

Characteristics[growth condition]
fetal enteroid grown medium NRG1, 100 nanogram per milliliter                                      5113
fetal enteroid grown medium EGF, 100 nanogram per milliliter                                       4004
fetal enteroid grown medium EGF, 100 nanogram per milliliter; NRG1, 100 nanogram per milliliter    1441
Name: count, dtype: int64

### Add age group

In [81]:
E8901.obs['developmental_stage'].value_counts()

developmental_stage
embryonic human stage                       49131
child stage                                 37839
9th week post-fertilization human stage     30502
10th week post-fertilization human stage     7559
adolescent stage                             6592
Name: count, dtype: int64

In [82]:
E9543.obs['developmental_stage'].value_counts()

developmental_stage
adult    62586
Name: count, dtype: int64

In [83]:
E9536.obs['developmental_stage'].value_counts()

developmental_stage
embryo    31494
Name: count, dtype: int64

In [84]:
E9489.obs['developmental_stage'].value_counts()

developmental_stage
late embryonic stage    223842
Name: count, dtype: int64

In [85]:
E9720.obs['developmental_stage'].value_counts()

developmental_stage
late embryonic stage    10558
Name: count, dtype: int64

In [86]:
for adata in anndata_objects:
    adata.obs['age_group'] = 'cell culture model'

    mask_primary = adata.obs['Characteristics[growth condition]'] == 'primary tissue'

    dev_stages = adata.obs.loc[mask_primary, 'developmental_stage']
    
    standard_stages = ['child stage', 'adolescent stage', 'adult']
    mask_standard = dev_stages.isin(standard_stages)
    
    adata.obs.loc[mask_primary & mask_standard, 'age_group'] = \
        adata.obs.loc[mask_primary & mask_standard, 'developmental_stage']
    
    mask_fetal = mask_primary & ~mask_standard
    
    adata.obs.loc[mask_fetal, 'age_group'] = \
        adata.obs.loc[mask_fetal, 'full_age'].apply(determine_trimester)

In [87]:
for adata in anndata_objects:
    print(f"\nDataset ID: {adata.obs['ArrayExpress_ID'].iloc[0]}")
    print("Age group distribution:")
    print(adata.obs['age_group'].value_counts())
    print('-' * 40)


Dataset ID: E-MTAB-8901
Age group distribution:
age_group
first trimester       63746
child stage           37839
cell culture model    15887
second trimester       7559
adolescent stage       6592
Name: count, dtype: int64
----------------------------------------

Dataset ID: E-MTAB-9489
Age group distribution:
age_group
second trimester    143385
first trimester      80457
Name: count, dtype: int64
----------------------------------------

Dataset ID: E-MTAB-9536
Age group distribution:
age_group
second trimester    20620
first trimester     10874
Name: count, dtype: int64
----------------------------------------

Dataset ID: E-MTAB-9543
Age group distribution:
age_group
adult    62586
Name: count, dtype: int64
----------------------------------------

Dataset ID: E-MTAB-9720
Age group distribution:
age_group
cell culture model    10558
Name: count, dtype: int64
----------------------------------------


### Add information about immunophenotype

In [88]:
E9543.obs['Characteristics[immunophenotype]'] = E9543.obs['Factor Value[immunophenotype]'].copy()
E9543.obs['Characteristics[immunophenotype]'].value_counts()

Characteristics[immunophenotype]
unsorted         28857
CD45 positive    19056
CD45 negative    14673
Name: count, dtype: int64

In [89]:
E8901.obs['Characteristics[immunophenotype]'].value_counts()

Characteristics[immunophenotype]
total cells       76118
EPCAM positive    49890
EPCAM negative     5615
Name: count, dtype: int64

In [90]:
E9489.obs['Characteristics[immunophenotype]'].value_counts()

Characteristics[immunophenotype]
unsorted     199211
live cell     24631
Name: count, dtype: int64

In [91]:
E9720.obs['Characteristics[immunophenotype]'] = 'unsorted'
E9536.obs['Characteristics[immunophenotype]'] = 'unsorted'

### Add donor_id

In [92]:
for adata in anndata_objects:
    adata.obs.rename(columns={'Characteristics[individual]': 'donor_id'}, inplace=True)

### Validate sex
+ Previously we identified sex during our quality metrics check by analysing the expression of XIST counts and Y chromosome genes. Some of our datasets already have this data. In the following code, we will compare sex predicted by us with the real one.

In [93]:
for adata in anndata_objects:
    if 'Characteristics[sex]' in adata.obs.columns:
        print(f"\nDataset ID: {adata.obs['ArrayExpress_ID'].iloc[0]}")
        
        true_sex = adata.obs['Characteristics[sex]'].astype(str)
        predicted_sex = adata.obs['sex'].astype(str)
        
        comparison = pd.DataFrame({
            'true_sex': true_sex,
            'predicted_sex': predicted_sex
        })
        
        print("\nOverall distribution:")
        print(pd.crosstab(comparison['true_sex'], comparison['predicted_sex']))
        
        mismatches = comparison[
            (comparison['true_sex'] != comparison['predicted_sex']) & 
            (~comparison['true_sex'].isin(['not available', 'NA', 'nan']))
        ]
        
        if len(mismatches) > 0:
            print(f"\nNumber of mismatches: {len(mismatches)}")
            print("\nMismatch distribution:")
            print(mismatches.groupby(['true_sex', 'predicted_sex']).size())
        else:
            print("\nNo mismatches found (excluding 'not available' cases)")
            
        print('-' * 50)
    else:
        print(f"\nDataset ID: {adata.obs['ArrayExpress_ID'].iloc[0]}")
        print("No Characteristics[sex] column present")
        print('-' * 50)


Dataset ID: E-MTAB-8901

Overall distribution:
predicted_sex  female   male
true_sex                    
female           8093      0
male                0  36338
not available   52310  34882

No mismatches found (excluding 'not available' cases)
--------------------------------------------------

Dataset ID: E-MTAB-9489

Overall distribution:
predicted_sex  female   male
true_sex                    
female         142731      0
male                0  81111

No mismatches found (excluding 'not available' cases)
--------------------------------------------------

Dataset ID: E-MTAB-9536
No Characteristics[sex] column present
--------------------------------------------------

Dataset ID: E-MTAB-9543

Overall distribution:
predicted_sex  female   male
true_sex                    
female          27828      0
male                0  34758

No mismatches found (excluding 'not available' cases)
--------------------------------------------------

Dataset ID: E-MTAB-9720
No Characteristics[se

### Delete columns we do not need

In [94]:
columns_to_delete = [
    'Characteristics[sex]', # we leave our column with predicted sex based on Y/XISTS counts. there is no mismatches with reported sex
    'Factor Value[immunophenotype]', # same as Characteristics[immunophenotype]
    'Factor Value[stimulus]', # same as Characteristics[growth condition]
    'Factor Value[age]', # same as Characteristics[age]
    'Factor Value[growth condition]' # same as Characteristics[growth condition]
]

In [95]:
for adata in anndata_objects:
    columns_to_drop = [col for col in columns_to_delete if col in adata.obs.columns]
    if columns_to_drop:
        adata.obs.drop(columns=columns_to_drop, inplace=True)

In [96]:
analysis_data = []

for col in adata.obs.columns:
    for adata in anndata_objects:
        dataset_id = adata.obs['ArrayExpress_ID'].iloc[0]
        
        if col in adata.obs.columns:
            unique_values = adata.obs[col].unique()
            example_values = adata.obs[col].iloc[:5]
            
            analysis_data.append({
                'Column': col,
                'Dataset_ID': dataset_id,
                'Present': 'Yes',
                'N_unique_values': len(unique_values),
                'Unique_values': ', '.join(map(str, unique_values)),
                'Example_values': ', '.join(map(str, example_values))
            })
        else:
            analysis_data.append({
                'Column': col,
                'Dataset_ID': dataset_id,
                'Present': 'No',
                'N_unique_values': 0,
                'Unique_values': 'Not present',
                'Example_values': 'N/A'
            })

analysis_df = pd.DataFrame(analysis_data)

In [97]:
for adata in anndata_objects:
    rename_dict = {}
    for col in adata.obs.columns:
        if (col in adata.obs.columns and 
            any(col.startswith(prefix) for prefix in ['Characteristics[', 'Factor Value[', ])):
            new_name = transform_column_name(col)
            rename_dict[col] = new_name
    if rename_dict:
        adata.obs.rename(columns=rename_dict, inplace=True)

### Save anndata objects

In [98]:
project = 'gut'
species = 'hs'
name = 'AM'
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')
counts = 'raw'

+ E-MTAB-8901

In [99]:
atribute = 'Elementaite2021_E-MTAB-8901'
current_history = E8901.uns['processing_history'].tolist()

new_entry = json.dumps({
    'timestamp': timestamp,
    'step': 'Metadata harmonized for integration with other datasets',
})
current_history.append(new_entry)

E8901.uns['processing_history'] = current_history

In [100]:
E8901.uns['processing_history']

['{"step": "added qc metrics (generated with sctk), doublets info, filtered cells with less than 100 genes", "timestamp": "25112024_121501"}',
 '{"timestamp": "02122024_143702", "step": "filtered cells based on consensus qc generated with sctk (default parameters), added sex covariates and cell cycle phase"}',
 '{"timestamp": "17122024_164056", "step": "Metadata harmonized for integration with other datasets"}']

In [ ]:
E8901.write_h5ad(f"Elementaite_2021/{project}_{species}_{atribute}_QC_filtered_metadata_harmonized_{name}_{timestamp}_{counts}.h5ad")

+ E-MTAB-9536

In [102]:
atribute = 'Elementaite2021_E-MTAB-9536'
current_history = E9536.uns['processing_history'].tolist()

new_entry = json.dumps({
    'timestamp': timestamp,
    'step': 'Metadata harmonized for integration with other datasets',
})
current_history.append(new_entry)

E9536.uns['processing_history'] = current_history

In [103]:
E9536.uns['processing_history']

['{"step": "create raw anndata after mapping, no filtering", "timestamp": "31102024_105424"}',
 '{"timestamp": "13112024_101436", "step": "added qc metrics (generated with sctk), doublets info, filtered cells with less than 100 genes"}',
 '{"timestamp": "13112024_104420", "step": "filtered cells based on consensus qc generated with sctk (default parameters), added sex covariates and cell cycle phase"}',
 '{"timestamp": "17122024_164056", "step": "Metadata harmonized for integration with other datasets"}']

In [ ]:
E9536.write_h5ad(f"Elementaite_2021/{project}_{species}_{atribute}_QC_filtered_metadata_harmonized_{name}_{timestamp}_{counts}.h5ad")

+ E-MTAB-9543

In [105]:
atribute = 'Elementaite2021_E-MTAB-9543'
current_history = E9543.uns['processing_history'].tolist()

new_entry = json.dumps({
    'timestamp': timestamp,
    'step': 'Metadata harmonized for integration with other datasets',
})
current_history.append(new_entry)

E9543.uns['processing_history'] = current_history

In [106]:
E9543.uns['processing_history']

['{"step": "create raw anndata after mapping, no filtering", "timestamp": "05112024_113505"}',
 '{"timestamp": "09122024_154541", "step": "doublets info; filtered cells with less than 100 genes; filtered cells: pct_mito <50, n_genes>50, n_counts>500, percent_hb<2, total_counts<9000, n_genes_by_counts<5000; added sex covariates and cell cycle phase"}',
 '{"timestamp": "17122024_164056", "step": "Metadata harmonized for integration with other datasets"}']

In [ ]:
E9543.write_h5ad(f"Elementaite_2021/{project}_{species}_{atribute}_QC_filtered_metadata_harmonized_{name}_{timestamp}_{counts}.h5ad")

+ E-MTAB-9489

In [108]:
atribute = 'Holloway2021_E-MTAB-9489'
current_history = E9489.uns['processing_history'].tolist()

new_entry = json.dumps({
    'timestamp': timestamp,
    'step': 'Metadata harmonized for integration with other datasets',
})
current_history.append(new_entry)

E9489.uns['processing_history'] = current_history

In [109]:
E9489.uns['processing_history']

['{"step": "added qc metrics (generated with sctk), doublets info, filtered cells with less than 100 genes", "timestamp": "27112024_122219"}',
 '{"timestamp": "09122024_100120", "step": "filtered cells based on consensus qc generated with sctk (default parameters), added sex covariates and cell cycle phase"}',
 '{"timestamp": "17122024_164056", "step": "Metadata harmonized for integration with other datasets"}']

In [ ]:
E9489.write_h5ad(f"Holloway_2021/{project}_{species}_{atribute}_QC_filtered_metadata_harmonized_{name}_{timestamp}_{counts}.h5ad")

+ E-MTAB-9720

In [111]:
atribute = 'Holloway2021_E-MTAB-9720'
current_history = E9720.uns['processing_history'].tolist()

new_entry = json.dumps({
    'timestamp': timestamp,
    'step': 'Metadata harmonized for integration with other datasets',
})
current_history.append(new_entry)

E9720.uns['processing_history'] = current_history

In [112]:
E9720.uns['processing_history']

['{"step": "create raw anndata after mapping, no filtering", "timestamp": "06112024_121025"}',
 '{"timestamp": "27112024_160858", "step": "added qc metrics (generated with sctk), doublets info, filtered cells with less than 100 genes"}',
 '{"timestamp": "09122024_103213", "step": "filtered cells based on consensus qc generated with sctk (default parameters), added sex covariates and cell cycle phase"}',
 '{"timestamp": "17122024_164056", "step": "Metadata harmonized for integration with other datasets"}']

In [ ]:
E9720.write_h5ad(f"Holloway_2021/{project}_{species}_{atribute}_QC_filtered_metadata_harmonized_{name}_{timestamp}_{counts}.h5ad")